# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SalehAl-Nassar/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=True)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I’m in Lane 2 — Refresh / Content Opportunity Scoring. That means my output is a **ranked queue** of pages, each with a priority score and one or more reason codes explaining why it scored that way. This is **scoring/ranking**, not classification, because:

- The reviewer doesn’t need a yes/no prediction (“page A is declining, page B is not”). They need a **priority order** — which page to look at first, second, third. A classifier gives labels; a scorer gives an ordering.
- The reason codes matter as much as the score. A page at rank 5 because it’s stale and visible needs a different response than a page at rank 5 because it’s declining with demand. Classification flattens that into one bit.
- Clustering doesn’t fit either — I’m not looking for groups of similar pages. I’m looking for a single actionable ordering across all pages.

So: scoring/ranking, with the metric chosen to match the decision (see section 3).

In [ ]:
# (Section 1 is text — see markdown cell above)

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My proxy label is **`is_declining`**, defined as `trend_direction == “down”`. This is available in the starter dataset and the warehouse. It splits the data roughly 54% positive / 46% negative — almost balanced.

I need to be clear about what this label **doesn’t** capture:

- It collapses magnitude. A page with trend_pct = -20% and one with -60% both get `down = True`, but they’re in very different situations. A future version should use the raw trend_pct or a bucketed score.
- It doesn’t account for pages that were already low-traffic. A page with 5 impressions going to 2 gets flagged as “declining”, but that’s noise, not a signal. The metric (section 3) will handle this by restricting to a minimum impressions floor.
- It’s a **current-window** label, not a future outcome. The lane guide (and the Week 1 notebook) flag this explicitly: `trend_direction` is computed from the same 90-day window, so using it as a label means the model learns to recognise the present, not predict the future. A stronger version will define decline as a future outcome (e.g. features from prior 90 days → decline over next 30 days) once I move to the warehouse.

For Week 2, this proxy is good enough to frame the task. I won’t mistake it for a final target.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'Shape: {df.shape}')
print(f'Unique content items: {df.content_id.nunique()}')
print(f'Unique clients: {df.client_id.nunique()}')
print()

# Proxy target: is this page currently declining?
df['is_declining_proxy'] = (df.trend_direction == 'down').astype(int)

print('=== Proxy target distribution ===')
vc = df['is_declining_proxy'].value_counts().sort_index()
for val, count in vc.items():
    label = 'declining' if val == 1 else 'not declining'
    pct = count / len(df) * 100
    print(f'  {label} ({val}): {count:>6} ({pct:.1f}%)')
print()
print(f'Base rate of positive class: {df.is_declining_proxy.mean() * 100:.1f}%')
print('(Classes are nearly balanced — no significant imbalance to correct for)')

Shape: (30000, 44)
Unique content items: 30000
Unique clients: 32

=== Proxy target distribution ===
  not declining (0):  13738 (45.8%)
  declining (1):  16262 (54.2%)

Base rate of positive class: 54.2%
(Classes are nearly balanced — no significant imbalance to correct for)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric I’ll defend is **Recall@K restricted to “visible” pages** (where visible means impressions_90d >= 500 or some similar floor I’ll set in the data contract). Here’s why:

- **Recall matters more than precision** because a false negative (a declining page the queue misses) is costlier than a false positive (a healthy page the reviewer looks at and dismisses in 10 minutes). A missed high-visibility decliner means real traffic keeps dropping.
- **Restricting to visible pages** avoids rewarding the model for correctly flagging low-traffic noise as “declining” — a page with 10 impressions that drops to 2 is technically declining but doesn’t need a reviewer’s time. The metric should measure whether the queue surfaces the pages that actually matter.
- **K should match reviewer capacity.** If a reviewer can audit 20 pages per cycle, the metric is Recall@20 (what fraction of all high-value decliners appear in the top 20?). If capacity is 50, it’s Recall@50.

I’ll also report Precision@K and Average Precision for completeness, but the primary number I’ll optimise for is Recall@K among visible pages. The baseline reference (hand-coded rules) scores 0.240 Precision@50 on the starter data (from the committed outputs/model_report.md). I want to see whether a model can improve the recall side without destroying precision.

In [ ]:
# (I'll compute the actual Recall@K baseline once the scoring function is built — Week 4's notebook)

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one **content page** (pseudonymized). The dataframe below shows the 19 columns most relevant to Lane 2: traffic and visibility metrics, trend indicators, content age/freshness, content depth, and engagement. Note that some columns have missing values — for example, row 3 in the printed dataframe shows NaN for word_count and word_count_tier (about 26% of rows are missing word_count in the full dataset). I’ll handle this in the signal audit with has_-flags rather than blind imputation, following the flyrank-data skill’s guidance. The full dataset has 44 columns; these are the ones I’d expect to use as features (plus the proxy target).

In [ ]:
# Re-using df from section 2 — same 30k-row slice, one row per content page
lane_cols = [
    'content_id', 'client_id',
    'impressions_90d', 'clicks_90d', 'avg_position', 'ctr',
    'trend_direction', 'trend_pct',
    'content_age_days', 'days_since_last_update', 'freshness_tier',
    'word_count', 'word_count_tier',
    'sessions_90d', 'engagement_rate', 'scroll_rate',
    'content_type', 'competition_level', 'main_intent'
]

lane_df = df[lane_cols].copy()
print(f'Shape: {lane_df.shape}  (one row = one content page)')
print()
print('=== First 5 rows ===')
print(lane_df.head().to_string())

Shape: (30000, 19)  (one row = one content page)

=== First 5 rows ===
             content_id          client_id  impressions_90d  clicks_90d  avg_position   ctr trend_direction  trend_pct  content_age_days  days_since_last_update freshness_tier  word_count word_count_tier  sessions_90d  engagement_rate  scroll_rate     content_type competition_level    main_intent
0  content_304f48230142  client_f369cb89fc             3803          29          10.6  0.76            down      -41.4               187                      20           0-30      3221.0       2000-3500            17             5.88         4.55  keyword article              HIGH  transactional
1  content_a1fb4e703a9e  client_4e07408562            15320           7          20.3  0.05            down      -57.7               445                      25           0-30      2481.0       2000-3500             9             0.00        10.00  keyword article               LOW  informational
2  content_9aa793d4d895  client_7f2

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule — like the baseline formula in the starter pipeline — works by setting fixed thresholds: “flag a page if impressions >= X AND trend == down AND position <= Y”. That works for the obvious cases, but it misses the messy ones because:

- **Decline looks different for high- vs low-traffic pages.** A page with 10K impressions dropping 20% is a bigger concern than a page with 100 impressions dropping 40%, but a fixed rule that treats both the same either generates too many false alarms or misses the high-value cases.
- **Multiple weak signals need to be weighted together.** A page might have stable impressions but declining CTR, old content, and weak engagement — none of those alone triggers a rule, but together they tell a story. A learned model can weight them; an if-statement can’t without an explosion of rules.
- **Interactions shift by content type and client.** The signals that predict decline for a keyword article might differ from those for a comparison article. A fixed rule that works for one client may not work for another. ML can learn per-group patterns from the data.

That said, the baseline is useful — it’s a transparent starting point. The question isn’t “rule vs ML”; it’s “can ML improve on the rule enough to justify the complexity?” The starter pipeline already suggests yes (Precision@50: 0.240 baseline vs 0.740 random forest, from the committed outputs/model_report.md). The 0.740 is the honest, leakage-corrected score — the pipeline uses a client-holdout split and excludes label-derived features, so no feature leakage inflates it.

In [ ]:
# (Section 5 is text — see markdown cell above)

## Self-check

Before you submit, confirm each line honestly:

- [X] **Task type named** — scoring/ranking, with a clear explanation of why not classification or clustering.
- [X] **Proxy limitation named** — `trend_direction == "down"` is a current-window label; collapses magnitude; flags low-traffic noise. Stronger version uses future outcomes.
- [X] **Metric named and defended** — Recall@K restricted to visible pages (impressions >= floor), because FNs cost more than FPs. Will also report Precision@K.
- [X] **Unit of analysis shown** — one row = one content page; 19-column slice of real data printed above.
- [X] **Why ML not a fixed rule explained** — decline signal differs by traffic volume; multiple weak signals interact; effects vary by content type and client.
- [X] **No client names, URLs, or private queries** — all IDs are pseudonymized.
- [X] **Notebook runs top to bottom** — all cells execute, all outputs visible.

Committed to `work/notebooks/w02_ml_task_framing.ipynb`. Done.